In [3]:
import numpy as np
import csv,os
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

#import data from CSVs
data = []
files=["../2_Database/TopBeerData.csv","../2_Database/GoodBeerData.csv","../2_Database/RestBeerData.csv"]
for i in range (0,len(files)):
    with open(files[i],"r",encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader) # remove header
        for row in reader:
            data.append(row)
print(len(data))
with open("BeerData.csv", 'w', newline='',encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['beer_text','brewery','beer'])
textual_data = []
with open("BeerData.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for row in data[1:]:
        beer_text = (
#            f"brewery: {row[2]}\n" #brewery removed since adds bias
            f"beer name: {row[3]} |"
            f"beer kind: {row[4]} |"
            f"description: {row[11]}|"
            f"ABV: {row[6]} |"
            f"IBU: {row[7]} |"
            f"rating: {row[8]}"
        )
        writer.writerow([beer_text,row[2],row[3]])
        textual_data.append(beer_text)
print("Adatok kiíratva CSV-be.")

2303
Adatok kiíratva CSV-be.


In [200]:
# ---------------------------------------------------
# generate embeddings with OpenAI - COSTS MONEY
# ---------------------------------------------------

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

embeddings = []
batch_size = 100
for i in range(0, len(textual_data), batch_size):
    batch = textual_data[i:i + batch_size]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend(
        item.embedding
        for item in response.data
    )

print(len(embeddings))

X_emb = np.array(embeddings)
print(X_emb.shape)

with open("BeerEmbeddings.csv", 'w', newline='',encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([""] * 1536) #1536 nr of coloumns  
    for row in X_emb:
        writer.writerow(row)
print("Embedding kiíratva CSVbe.")

2302
(2302, 1536)
Embedding kiíratva CSVbe.


In [19]:
from difflib import SequenceMatcher

beer_data=[]
with open("BeerData.csv","r",encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader) # remove header
    for row in reader:
        beer_data.append(row)

#get user beer preference
pref_beer = input("Add meg a sör nevét: ")

best_sim=0
count=0
for i, text in enumerate(beer_data):
    if pref_beer.lower() == text[2].lower():
        print(i,text[2],"Full matching of the beer name!!! \n")
        count=1
        best_sim=1
        best_idx = i
        best_text=text[2]
        best_brewery=text[1]
        break
    else:
        sim = SequenceMatcher(None, text[2].lower(), pref_beer.lower()).ratio()  
        if sim > best_sim:
            best_sim = sim
            best_idx = i
            best_text=text[2]
            best_brewery=text[1]
    
if (count==0 and best_sim<0.5) :     #0.5 is arbitrary
    print("nem találtam a kiválasztott sört")
else:
    print(f"Index: {best_idx}")
    print(f"Similarity: {best_sim:.3f}")
    print(f"A kiválasztott sör neve: {best_text}")
    print(f"A kiválasztott sör gyártója: {best_brewery}")
    pref_beer=best_text

Add meg a sör nevét:  fisher düppel


Index: 423
Similarity: 0.846
A kiválasztott sör neve: Fisher Dübbel
A kiválasztott sör gyártója: CSAPON! Podcast


In [20]:
#CSV reading
emb_data=[]
with open("BeerEmbeddings.csv","r",encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader) # remove header
    for row in reader:
        emb_data.append(row)
    
    #cosine similarity
    emb_data = np.array(emb_data, dtype=np.float32) #necessary conversion
    
    pref = emb_data[best_idx].reshape(1, -1)
    similarities = cosine_similarity(pref, emb_data)[0]
    top_k = 5
    top_idx = np.argsort(similarities)[::-1] 
    top_idx = top_idx[top_idx != (best_idx)][:top_k]
    print(top_idx,"\n")
    for i in top_idx:
        print(i, similarities[i],beer_data[i])
        print("\n")


[ 426  424  439 1989 2254] 

426 0.7549567 ['beer name: Dubbel |beer kind: Belgian Dubbel |description: Visszafogottan komlózott keserűsége teret enged a belga kandiscukor karamellás édességének és a malátás ízjegyeknek. Az élesztőnek köszönhetően jellegzetes banános, mazsolás illatok társulnak a krémes habhoz, melynek egy egyhén alkoholos, kerek kortyérzeteteredményeznek. Fogyasztása 10-14 °C-on javasolt! |ABV: 6.5 |IBU: 20 |rating: 3.53', 'Pannonhalmi Főapátság Sörfőzde', 'Dubbel']


424 0.7027572 ['beer name: TesztAndrás Omega.02 |beer kind: Belgian Dubbel |description: Az Omega.02 az apátsági sörgyártás szentandrási víziója, ami a belgák egyik legkomolyabb élesztőjével, a Lallemand Abbaye-al készült. Ebben a visszafogott komlózottságú barna dubbelben sem maradhat el a tradicionális belga párjára jellemző karakteres fűszeresség, a selymes korty és az intenzív mazsolás, datolyás illatok. Komoly beltartalmú, impozáns őszi sör, ahogy azt Belgiumban megálmodták. |ABV: 7.0 |IBU: 25 |rati